In [1]:
# merge the two tables
import pandas as pd
train_transaction = pd.read_csv('rawdata/train_transaction.csv')
train_identity = pd.read_csv('rawdata/train_identity.csv')
train_combined = train_transaction.merge(train_identity, on='TransactionID', how='left')

In [ ]:
# convert TransactionDT to a day number
## TransactionDT is in seconds - divide by 86400 to get days
train_combined['TransactionDay'] = (train_combined['TransactionDT'] / 86400).astype(int)
## inspect
print(train_combined['TransactionDay'].describe())

count    590540.000000
mean         84.729199
std          53.437277
min           1.000000
25%          35.000000
50%          84.000000
75%         130.000000
max         182.000000
Name: TransactionDay, dtype: float64


C:\Users\minim\AppData\Local\Temp\ipykernel_21032\3474287003.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_combined['TransactionDay'] = (train_combined['TransactionDT'] / 86400).astype(int)


In [4]:
# compute D1n (the card's birthday)
## D1n = TransactionDay - D1
train_combined['D1n'] = train_combined['TransactionDay'] - train_combined['D1']
## inspect - does this look stable for the same card
print(train_combined[['card1', 'D1', 'TransactionDay', 'D1n']].head(20))

    card1     D1  TransactionDay    D1n
0   13926   14.0               1  -13.0
1    2755    0.0               1    1.0
2    4663    0.0               1    1.0
3   18132  112.0               1 -111.0
4    4497    0.0               1    1.0
5    5937    0.0               1    1.0
6   12308    0.0               1    1.0
7   12695    0.0               1    1.0
8    2803    0.0               1    1.0
9   17399   61.0               1  -60.0
10  16496    1.0               1    0.0
11   4461    0.0               1    1.0
12   3786   72.0               1  -71.0
13  12866   46.0               1  -45.0
14  11839    0.0               1    1.0
15   7055    0.0               1    1.0
16   1790    0.0               1    1.0
17  11492    0.0               1    1.0
18   4663    0.0               1    1.0
19   7005   62.0               1  -61.0


In [ ]:
## group by 'card1' and see if 'D1n' is consistent
### group and aggregate
consistency_check = train_combined.groupby('card1').agg(
    total_transactions=('D1n', 'count'),
    unique_d1n_count=('D1n', 'nunique')
).reset_index()

### isolate cards that have more than 5 transactions
multi_tx_cards = consistency_check[consistency_check['total_transactions'] > 5].copy()

### how many cards have consisten D1n value counts
perfect_cards = multi_tx_cards[multi_tx_cards['unique_d1n_count'] == 1]

### calculate to see if the plan holds up
total_multi_tx = len(multi_tx_cards)
total_perfect = len(perfect_cards)

if total_multi_tx > 0:
    consistency_rate = (total_perfect / total_multi_tx) * 100
else:
    consistency_rate = 0.0

### summary
print(f"Total cards with more than 5 transactions: {total_multi_tx}")
print(f"Cards with a perfectly consistent D1n: {total_perfect}")
print(f"Consistency Rate: {consistency_rate:.2f}%\n")

### if the rate isn't 100% -> look at the cards where D1n varies the most
if consistency_rate < 100:
    print("Here are the cards where the insight breaks down the most (highest D1n variant):")
    inconsistent_cards = multi_tx_cards.sort_values(by='unique_d1n_count', ascending=False)
    print(inconsistent_cards.head(10))

# SUMMARY: Not a good outcome!
# TODO: find out why it's not a good outcome

Total cards with more than 5 transactions: 5883
Cards with a perfectly consistent D1n: 320
Consistency Rate: 5.44%

Here are the cards where the insight breaks down the most (highest D1n variant):
       card1  total_transactions  unique_d1n_count
6615    9500               14129               591
12616  17188               10325               571
5365    7919               14920               541
9143   12695                7081               532
1412    2803                6125               512
9014   12544                6761               501
11593  15885               10332               485
10950  15066                7920               469
9252   12839                5117               444
7950   11207                3687               437


In [18]:
# build the UID
train_combined['UID'] = (
    train_combined['card1'].astype(str) + '_' +
    train_combined['addr1'].astype(str) + '_' +
    train_combined['D1n'].astype(str)
)

print(train_combined['UID'].unique()) # how many unique "cards" did we find
print(train_combined['UID'].value_counts()[:10]) # most frequent UIDs

C:\Users\minim\AppData\Local\Temp\ipykernel_21032\4211284080.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_combined['UID'] = (


<StringArray>
[ '13926_315.0_-13.0',     '2755_325.0_1.0',     '4663_330.0_1.0',
 '18132_476.0_-111.0',     '4497_420.0_1.0',     '5937_272.0_1.0',
    '12308_126.0_1.0',    '12695_325.0_1.0',     '2803_337.0_1.0',
  '17399_204.0_-60.0',
 ...
  '15813_441.0_182.0',  '8431_325.0_-443.0',  '10640_330.0_182.0',
  '16873_110.0_182.0',   '2198_315.0_182.0',   '7826_387.0_182.0',
 '11942_310.0_-451.0', '12037_231.0_-133.0',  '10444_204.0_182.0',
  '12037_231.0_182.0']
Length: 199071, dtype: str
UID
15775_330.0_129.0     1414
9500_126.0_-85.0       446
8900_231.0_-60.0       232
8528_387.0_-159.0      215
12741_143.0_-202.0     196
7207_204.0_-465.0      191
13597_191.0_-48.0      148
4121_476.0_-8.0        142
12695_325.0_-342.0     123
2884_204.0_-32.0       118
Name: count, dtype: int64


In [19]:
# sanity check on the UID
## do fraudulent transactions cluster in specific UIDs?
fraud_by_uid = train_combined.groupby('UID')['isFraud'].agg(['sum', 'count', 'mean'])
fraud_by_uid.columns = ['fraud_count', 'total_txns', 'fraud_rate']
fraud_by_uid = fraud_by_uid.sort_values('fraud_rate', ascending=False)
print(fraud_by_uid.head(20))

                    fraud_count  total_txns  fraud_rate
UID                                                    
16659_264.0_5.0               3           3         1.0
13844_472.0_75.0              1           1         1.0
18097_126.0_107.0             1           1         1.0
15066_123.0_2.0               1           1         1.0
6957_310.0_44.0               1           1         1.0
10289_325.0_67.0              1           1         1.0
6957_441.0_173.0              3           3         1.0
6958_472.0_167.0              1           1         1.0
16346_284.0_138.0             1           1         1.0
13844_476.0_57.0              1           1         1.0
16075_299.0_167.0             1           1         1.0
14869_337.0_136.0             1           1         1.0
12695_126.0_-184.0           15          15         1.0
1675_110.0_123.0              1           1         1.0
14098_325.0_111.0             3           3         1.0
15863_428.0_28.0              1           1     